# Lab 13: When Do Investors Sell, and What Does It Cost Them?

*Financial Engineering and the Evidence*, Chapter 13.

**Claim on trial:** investors earn the returns their stocks earn, whatever rule they use to decide when to sell.

You will apply six selling rules to the same prices, first in simulation and then on a century of industry returns, and see what each rule does to the investor's experience, to the return per year invested, and to the investor's wealth after tax. Before each step, write down a prediction.

In [ ]:
from pathlib import Path
from IPython.display import Image, display
import lab13

SOURCE = "french"   # "french" downloads real data; "sim" uses fake industry returns to test the code
UP, DOWN = 0.2, 0.2 # profit target and stop-loss
CAP = 60            # longest hold on real data, in months
SEED = 2026
OUT = Path("report")
OUT.mkdir(exist_ok=True)

df = lab13.load_industries(SOURCE)
SIM = df.attrs.get("simulated", False)
PERIOD = f"{df.index[0]} to {df.index[-1]}"
print(PERIOD, "|", df.shape[1], "industries | simulated:", SIM)

## Part A: Optional stopping

The simulated prices are a fair game: their expected value next month is today's price.

**Predict first:** which rule will have the highest win rate? Will any rule have a mean return different from zero?

In [ ]:
a = lab13.part_a(50000, 0.30, 120, UP, DOWN, SEED)
lab13.fig_distributions(a, OUT, 0.30, 120)
display(Image(OUT / "fig13_1_distributions.png"))
a[["rule", "mean_hpr", "se_mean_hpr", "median_hpr", "win_rate", "mean_loss_if_loss", "share_losers_at_cap"]].round(4)

## Part B: Comparing returns over random horizons

Now the price drifts upward by 8% a year.

**Predict first:** a rule that sells at +20% after a few months will look spectacular if each sale is annualized on its own. Will it also earn more per year invested?

In [ ]:
b = lab13.part_b(50000, 0.08, 0.30, 120, UP, DOWN, SEED)
lab13.fig_per_year(b, OUT, 0.08, 0.30)
display(Image(OUT / "fig13_2_per_year.png"))
b[["rule", "mean_months", "mean_hpr", "naive_annual", "per_year_log", "wald_benchmark"]].round(4)

## Part C: Real prices

Each rule is applied to a position bought every month in each of the 49 industries.

**Predict first:** real industry returns show momentum over a year and some reversal over longer horizons. Which rule do you expect to fall short of the industries' average return per year invested?

In [ ]:
c = lab13.part_c(df, CAP, UP, DOWN, n_boot=400, seed=SEED)
lab13.fig_real(c["table"], OUT, SIM, PERIOD, CAP)
display(Image(OUT / "fig13_3_real_prices.png"))
print(f"Industries' average log return: {c['avg_log_per_year']:.2%} a year")
c["table"][["rule", "positions", "mean_months", "win_rate", "per_year_log", "excess_per_year", "t"]].round(4)

## Part D: The disposition test

**Predict first:** do the winners that a disposition investor sells go on to earn more or less over the next year than the losers the investor keeps?

In [ ]:
d = lab13.part_d(c["pos"], CAP, UP)
dec = lab13.by_decade(d, df.index)
lab13.fig_disposition(d, dec, OUT, SIM, PERIOD, UP)
display(Image(OUT / "fig13_4_disposition.png"))
print(f"Winners sold {d['sold_winners_mean']:+.2%}, losers held {d['held_losers_mean']:+.2%}, "
      f"gap {d['diff_mean']:+.2%} (Newey-West t = {d['diff_t']:.2f})")
dec.round(4)

## Part E: Taxes and costs

**Predict first:** rank buy-and-hold, selling every year, selling winners and selling losers by after-tax wealth.

In [ ]:
e = lab13.part_e(30, 20000, 0.08, 0.30, 0.20, 0.001, UP, DOWN, SEED)
lab13.fig_taxes(e, OUT, 0.20, 0.001, 0.08, 0.30, 30)
display(Image(OUT / "fig13_5_taxes.png"))
e.drop(columns="kind").round(4)

## Your verdict

1. Which parts of the claim survive: the expected return, the experience, the return per year invested, the return after tax?
2. On real prices, which rule fell furthest from the industries' average, and what property of returns explains it?
3. Write a one-sentence verdict on the scale of Chapter 1 (Survives, Weakens, Fails, Provisional), with the sample you tested.